# Benchmark results

Tables from the outputs of `run_benchmark.py` (see `benchmark.py` for the protocol). To reproduce:

```
python run_benchmark.py impute --horizon 6
python run_benchmark.py fit --horizon 6 --model all
python run_benchmark.py impute --horizon 8
python run_benchmark.py fit --horizon 8 --model all
```

Each model's outputs are in `results/h{6,8}/{model}/`: `metrics.csv`, `selection.json`, `validation_grid.csv`, and the per-person predictions (not published).
Populations: `all` = every participant (missing predictors imputed by the training-only imputer; primary), `complete` = participants with complete predictors (sensitivity).

In [1]:
import glob
import json
import os

import pandas as pd

MODELS = ['LR', 'GNB', 'DT', 'LightGBM', 'CatBoost', 'RealMLP', 'TabICL', 'GAT']
METRICS = ['AUROC', 'AUPRC', 'Precision', 'Recall', 'F1', 'Accuracy', 'Brier']

frames = []
for path in glob.glob('results/h*/*/metrics.csv'):
    horizon, model = path.split(os.sep)[-3:-1]
    frames.append(pd.read_csv(path).assign(horizon=int(horizon[1:]), model=model))
metrics = pd.concat(frames, ignore_index=True)
metrics['model'] = pd.Categorical(metrics['model'], MODELS, ordered=True)
print(metrics.groupby(['horizon', 'model'], observed=True).size())

horizon  model   
6        LR          22
         GNB         22
         DT          22
         LightGBM    22
         CatBoost    22
         RealMLP     22
         TabICL      22
         GAT         22
8        LR          22
         GNB         22
         DT          22
         LightGBM    22
         CatBoost    22
         RealMLP     22
         TabICL      22
         GAT         22
dtype: int64


## Internal validation: mean (SD) over 10 repeats

In [2]:
def mean_sd(df):
    g = df.groupby(['horizon', 'population', 'model'], observed=True)[METRICS]
    m, s = g.mean() * 100, g.std() * 100
    out = m.round(1).astype(str) + ' (' + s.round(1).astype(str) + ')'
    out['Brier'] = (m['Brier'] / 100).round(3).astype(str) + ' (' + (s['Brier'] / 100).round(3).astype(str) + ')'
    return out

internal = mean_sd(metrics[metrics['setting'] == 'internal'])
internal

AUROC       AUPRC    Precision       Recall  \
horizon population model                                                        
6       all        LR        85.7 (1.0)  50.4 (2.7)   45.0 (2.9)   54.5 (5.8)   
                   GNB       79.1 (1.5)  32.7 (1.8)   27.8 (2.3)   62.2 (7.2)   
                   DT        79.6 (0.9)  38.7 (1.4)   35.9 (5.6)   53.4 (9.2)   
                   LightGBM  86.1 (0.9)  51.9 (2.6)   45.3 (3.2)   56.4 (7.2)   
                   CatBoost  86.1 (0.7)  52.1 (1.9)   44.6 (3.1)   56.4 (6.0)   
                   RealMLP   85.6 (1.2)  50.6 (2.4)   43.5 (6.8)   58.7 (7.8)   
                   TabICL    86.6 (0.9)  52.5 (2.6)   47.0 (3.2)   54.0 (4.8)   
                   GAT       83.9 (2.1)  46.9 (5.1)  43.3 (14.7)  54.8 (19.6)   
        complete   LR        84.9 (1.0)  44.5 (2.7)   42.0 (2.8)   48.6 (6.6)   
                   GNB       78.4 (1.4)  28.0 (1.6)   24.6 (2.1)   59.4 (7.1)   
                   DT        79.1 (1.2)  34.0 (1.7)   32.5 (6.4)   50.8 (9.0)   
                   LightGBM  85.5 (0.9)  46.6 (2.3)   42.8 (3.8)   51.7 (7.6)   
                   CatBoost  85.4 (0.7)  46.3 (2.3)   41.8 (3.3)   51.8 (6.3)   
                   RealMLP   84.8 (1.0)  44.8 (2.4)   40.6 (7.1)   54.2 (8.1)   
                   TabICL    86.0 (0.8)  46.6 (2.5)   43.9 (2.8)   48.5 (5.7)   
                   GAT       83.0 (2.2)  41.1 (4.5)  40.1 (14.3)  50.1 (21.1)   
8       all        LR        87.1 (1.1)  61.5 (3.1)   52.5 (2.5)   64.3 (3.8)   
                   GNB       80.4 (1.0)  40.4 (2.7)   35.9 (2.0)   67.7 (4.7)   
                   DT        82.0 (1.0)  50.2 (2.6)   45.2 (6.2)   60.6 (6.6)   
                   LightGBM  87.1 (0.9)  61.8 (2.6)   52.8 (2.3)   62.9 (3.4)   
                   CatBoost  87.4 (1.0)  62.3 (2.8)   54.2 (4.8)   61.5 (7.9)   
                   RealMLP   86.7 (1.3)  60.7 (3.5)   47.0 (4.2)   69.6 (6.2)   
                   TabICL    87.7 (0.9)  62.4 (3.0)   53.5 (5.1)   63.9 (6.7)   
                   GAT       85.3 (1.5)  57.0 (4.6)  53.8 (12.2)  55.1 (16.4)   
        complete   LR        86.5 (1.4)  56.3 (5.0)   50.1 (3.1)   60.1 (4.9)   
                   GNB       79.9 (1.2)  36.0 (3.2)   32.4 (2.2)   65.3 (5.5)   
                   DT        81.6 (1.5)  46.4 (3.8)   41.6 (6.8)   58.9 (6.2)   
                   LightGBM  86.6 (1.3)  57.5 (4.6)   50.3 (3.3)   59.5 (4.2)   
                   CatBoost  86.8 (1.3)  57.4 (4.8)   51.8 (5.8)   57.4 (8.3)   
                   RealMLP   86.2 (1.4)  56.0 (5.2)   44.2 (3.5)   65.8 (7.3)   
                   TabICL    87.1 (1.2)  57.8 (4.9)   51.1 (4.9)   60.2 (8.1)   
                   GAT       84.7 (1.6)  52.5 (5.1)  50.8 (12.9)  51.9 (17.1)   

                                     F1    Accuracy          Brier  
horizon population model                                            
6       all        LR        49.0 (1.8)  86.8 (0.9)  0.077 (0.002)  
                   GNB       38.2 (2.0)  76.4 (3.5)  0.144 (0.003)  
                   DT        41.9 (1.4)  82.6 (3.6)  0.086 (0.001)  
                   LightGBM  49.8 (2.2)  86.8 (1.1)  0.076 (0.002)  
                   CatBoost  49.5 (1.4)  86.6 (1.1)  0.076 (0.002)  
                   RealMLP   49.1 (2.4)  85.7 (2.5)  0.079 (0.003)  
                   TabICL    50.0 (1.7)  87.4 (1.0)  0.076 (0.002)  
                   GAT       43.5 (3.9)  83.3 (6.7)  0.203 (0.036)  
        complete   LR        44.7 (2.3)  88.0 (0.8)  0.071 (0.003)  
                   GNB       34.6 (1.8)  77.2 (3.6)  0.135 (0.005)  
                   DT        38.5 (2.0)  83.4 (3.8)  0.078 (0.002)  
                   LightGBM  46.2 (2.3)  88.0 (1.2)   0.07 (0.002)  
                   CatBoost  45.9 (1.9)  87.8 (1.1)   0.07 (0.002)  
                   RealMLP   45.4 (2.0)  86.8 (2.4)  0.073 (0.003)  
                   TabICL    45.8 (1.6)  88.5 (0.8)   0.07 (0.003)  
                   GAT       39.0 (4.7)  84.4 (6.6)  0.196 (0.035)  
8       all        LR        57.7 (1.

## Temporal validation (2010 cohort)

In [3]:
temporal = metrics[metrics['setting'] == 'temporal'].set_index(['horizon', 'population', 'model'])
temporal_table = temporal[METRICS].copy()
temporal_table[METRICS[:-1]] = (temporal_table[METRICS[:-1]] * 100).round(1)
temporal_table['Brier'] = temporal_table['Brier'].round(3)
temporal_table[['n', 'cases']] = temporal[['n', 'cases']]
temporal_table.sort_index()

AUROC  AUPRC  Precision  Recall    F1  Accuracy  \
horizon population model                                                       
6       all        LR         83.8   46.8       45.1    49.0  46.9      87.2   
                   GNB        77.1   27.9       25.0    57.7  34.9      75.1   
                   DT         78.2   35.2       42.7    41.4  42.0      86.8   
                   LightGBM   84.4   47.6       44.0    49.6  46.6      86.8   
                   CatBoost   84.7   48.1       43.4    51.7  47.2      86.6   
                   RealMLP    84.7   47.1       45.0    44.7  44.9      87.3   
                   TabICL     85.1   48.8       46.5    48.4  47.5      87.6   
                   GAT        82.8   44.3       33.7    66.0  44.6      81.0   
        complete   LR         83.1   37.5       40.0    41.8  40.9      88.7   
                   GNB        76.1   21.9       20.5    52.2  29.5      76.7   
                   DT         76.8   27.8       35.9    34.5  35.2      88.1   
                   LightGBM   83.5   38.4       38.2    42.1  40.1      88.2   
                   CatBoost   83.9   39.1       37.8    44.4  40.8      88.0   
                   RealMLP    84.2   38.5       40.2    37.7  38.9      88.9   
                   TabICL     84.4   39.7       41.4    41.4  41.4      89.1   
                   GAT        82.2   36.2       29.4    61.5  39.8      82.6   
8       all        LR         84.7   57.1       51.9    57.4  54.5      84.7   
                   GNB        77.7   36.5       32.6    64.4  43.3      73.1   
                   DT         79.7   46.2       40.7    62.6  49.3      79.5   
                   LightGBM   85.1   58.0       49.9    60.2  54.5      84.0   
                   CatBoost   85.5   58.6       49.8    60.4  54.6      84.0   
                   RealMLP    85.5   57.8       53.0    55.2  54.1      85.1   
                   TabICL     86.0   59.8       52.9    59.1  55.8      85.1   
                   GAT        83.9   55.9       58.2    45.3  51.0      86.1   
        complete   LR         84.2   48.8       46.9    51.4  49.0      86.0   
                   GNB        77.2   30.2       28.1    60.0  38.2      74.6   
                   DT         78.7   38.3       35.9    57.7  44.3      81.0   
                   LightGBM   84.5   50.0       45.0    54.2  49.2      85.3   
                   CatBoost   84.8   50.4       44.3    54.0  48.7      85.1   
                   RealMLP    85.0   49.8       47.9    49.2  48.5      86.3   
                   TabICL     85.4   52.0       48.1    53.5  50.6      86.3   
                   GAT        83.6   48.3       53.1    39.4  45.3      87.5   

                             Brier      n  cases  
horizon population model                          
6       all        LR        0.080  14659   1699  
                   GNB       0.158  14659   1699  
                   DT        0.088  14659   1699  
                   LightGBM  0.079  14659   1699  
                   CatBoost  0.079  14659   1699  
                   RealMLP   0.080  14659   1699  
                   TabICL    0.079  14659   1699  
                   GAT       0.231  14659   1699  
        complete   LR        0.070  11835   1106  
                   GNB       0.143  11835   1106  
                   DT        0.077  11835   1106  
                   LightGBM  0.070  11835   1106  
                   CatBoost  0.069  11835   1106  
                   RealMLP   0.070  11835   1106  
                   TabICL    0.069  11835   1106  
                   GAT       0.219  11835   1106  
8       all        LR        0.097  12564   2000  
                   GNB       0.177  12564   2000  
                   DT        0.108  12564   2000  
                   LightGBM  0.096  12564   2000  
                   CatBoost  0.096  12564   2000  
                   RealMLP   0.098  12564   2000  
                   TabICL    0.094  12564   2000  
                   GAT       0.18

## Selected hyperparameters and thresholds

In [4]:
rows = []
for path in glob.glob('results/h*/*/selection.json'):
    sel = json.load(open(path))
    picks = pd.Series([str(r['params']) for r in sel['per_repeat']]).value_counts()
    rows.append({'horizon': sel['horizon'], 'model': sel['model'],
                 'temporal configuration': str(sel['temporal']['params']),
                 'temporal threshold': round(sel['temporal']['threshold'], 3),
                 'internal selections (count of 10)': '; '.join(f'{k}: {v}' for k, v in picks.items())})
pd.DataFrame(rows).sort_values(['horizon', 'model'])

,horizon,model,temporal configuration,temporal threshold,internal selections (count of 10)
0,6,CatBoost,"{'iterations': 200, 'learning_rate': 0.05, 'de...",0.223,"{'iterations': 200, 'learning_rate': 0.05, 'de..."
1,6,DT,"{'max_depth': 5, 'min_samples_split': 10, 'min...",0.215,"{'max_depth': 5, 'min_samples_split': 2, 'min_..."
2,6,GAT,"{'learning_rate': 0.01, 'dropout_p': 0.5}",0.727,"{'learning_rate': 0.01, 'dropout_p': 0.5}: 6; ..."
3,6,GNB,"{'var_smoothing': 0.0001, 'priors': None}",0.001,"{'var_smoothing': 0.0001, 'priors': None}: 8; ..."
5,6,LR,"{'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}",0.239,"{'C': 1, 'penalty': 'l1', 'solver': 'liblinear..."
4,6,LightGBM,"{'num_leaves': 20, 'max_depth': 5, 'learning_r...",0.242,"{'num_leaves': 20, 'max_depth': 5, 'learning_r..."
6,6,RealMLP,"{'batch_size': 256, 'n_epochs': 100}",0.244,"{'batch_size': 256, 'n_epochs': 100}: 4; {'bat..."
7,6,TabICL,"{'n_estimators': 16, 'batch_size': 8}",0.241,"{'n_estimators': 16, 'batch_size': 8}: 5; {'n_..."
8,8,CatBoost,"{'iterations': 200, 'learning_rate': 0.05, 'de...",0.242,"{'iterations': 200, 'learning_rate': 0.05, 'de..."
9,8,DT,"{'max_depth': 5, 'min_samples_split': 2, 'min_...",0.244,"{'max_depth': 5, 'min_samples_split': 2, 'min_..."


## Temporal validation by overlap with the development cohort

HRS follows the same people and households, so part of the 2010 cohort was in the 2000 development cohort. Performance is shown separately for returning participants, new participants from a development household, and new participants from new households (independent of the development data at both the participant and household level). Primary population (`all`).

In [5]:
import benchmark
from cohort_tables import overlap_status
from sklearn.metrics import average_precision_score, roc_auc_score

rows = []
for horizon in (6, 8):
    dev, ext, P = benchmark.load_cohorts(horizon)
    for model in MODELS:
        path = f'results/h{horizon}/{model}/temporal_predictions.csv'
        if not os.path.exists(path):
            continue
        pred = pd.read_csv(path)
        pred['overlap'] = overlap_status(pred['hhidpn'], dev['hhidpn'])
        for group, d in [('all', pred)] + list(pred.groupby('overlap')):
            rows.append({'horizon': horizon, 'model': model, 'group': group, 'n': len(d), 'cases': int(d['y'].sum()),
                         'AUROC': round(roc_auc_score(d['y'], d['p']) * 100, 1),
                         'AUPRC': round(average_precision_score(d['y'], d['p']) * 100, 1)})
overlap = pd.DataFrame(rows)
overlap.pivot_table(index=['horizon', 'model'], columns='group', values=['AUROC', 'AUPRC'])

AUPRC                                \
group              all new participant and household   
horizon model                                          
6       CatBoost  48.1                          36.8   
        DT        35.2                          21.5   
        GAT       44.3                          33.8   
        GNB       27.9                          19.1   
        LR        46.8                          35.8   
        LightGBM  47.6                          35.9   
        RealMLP   47.1                          35.5   
        TabICL    48.8                          38.1   
8       CatBoost  58.6                          45.8   
        DT        46.2                          31.5   
        GAT       55.9                          43.7   
        GNB       36.5                          24.3   
        LR        57.1                          43.7   
        LightGBM  58.0                          45.5   
        RealMLP   57.8                          44.6   
        TabICL    59.8                          47.4   

                                                                               \
group            new participant, development household returning participant   
horizon model                                                                   
6       CatBoost                                   56.6                  52.7   
        DT                                         39.8                  40.1   
        GAT                                        52.9                  48.6   
        GNB                                        34.2                  34.2   
        LR                                         58.2                  51.2   
        LightGBM                                   55.4                  52.4   
        RealMLP                                    55.6                  51.4   
        TabICL                                     58.6                  53.2   
8       CatBoost                                   62.5                  63.9   
        DT                                         47.5                  51.3   
        GAT                                        62.3                  60.6   
        GNB                                        42.3                  45.0   
        LR                                         63.2                  62.3   
        LightGBM                                   62.8                  63.3   
        RealMLP                                    62.9                  62.9   
        TabICL                                     65.1                  64.9   

                 AUROC                                \
group              all new participant and household   
horizon model                                          
6       CatBoost  84.7                          84.7   
        DT        78.2                          74.4   
        GAT       82.8                          82.5   
        GNB       77.1                          77.1   
        LR        83.8                          83.5   
        LightGBM  84.4                          84.9   
        RealMLP   84.7                          84.6   
        TabICL    85.1                          85.3   
8       CatBoost  85.5                          85.4   
        DT        79.7                          76.4   
        GAT       83.9                          83.0   
        GNB       77.7                          77.2   
        LR        84.7                          83.9   
        LightGBM  85.1                          85.3   
        RealMLP   85.5                          85.0   
        TabICL    86.0                          85.8   

                                                                               
group            new participant, development household returning participant  
horizon model                                                                  
6       CatBoost                                   86.5                  83.4  
        DT                 

## Cohort tables (from `cohort_tables.py`)

In [6]:
for name in ['participant_flow', 'complete_vs_incomplete', 'cohort_overlap']:
    display(pd.read_csv(f'results/tables/{name}.csv'))

,cohort,horizon (years),interviewed at baseline,excluded: age <= 50,excluded: dementia at baseline,excluded: baseline dementia status unknown,eligible,"excluded: died before final wave, no dementia observed","excluded: alive or unknown, not assessed at final wave",analytic cohort,incident dementia,complete predictors,"incident dementia, complete predictors"
0,2000 baseline,6,19578,707,1512,23,17336,2467,1211,13658,1593,11517,1148
1,2010 baseline,6,22034,1340,1445,6,19243,2398,2186,14659,1699,11835,1106
2,2000 baseline,8,19578,707,1512,23,17336,3167,1227,12942,2015,10911,1475
3,2010 baseline,8,22034,1340,1445,6,19243,3112,3567,12564,2000,10199,1337


,cohort,horizon (years),characteristic,complete predictors,incomplete predictors,standardized difference,n complete,n incomplete
0,2000 baseline,6,incident dementia (%),9.97,20.78,0.303,11517,2141
1,2000 baseline,6,age (years),65.91,66.58,0.071,11517,2141
2,2000 baseline,6,female (%),61.05,47.17,-0.281,11517,2141
3,2000 baseline,6,proxy interview (%),0.00,40.92,1.177,11517,2141
4,2000 baseline,6,non-Hispanic Black (%),11.04,17.94,0.197,11517,2141
5,2000 baseline,6,Hispanic (%),6.75,12.57,0.198,11517,2141
6,2000 baseline,6,education 13+ years (%),41.15,30.17,-0.231,11517,2141
7,2000 baseline,6,fair or poor self-rated health (%),20.77,31.13,0.238,11517,2141
8,2000 baseline,6,any ADL difficulty (%),11.39,16.27,0.142,11517,2141
9,2000 baseline,6,number of mobility difficulties (0-5),0.81,1.07,0.197,11517,2141


,horizon (years),2010 analytic cohort,returning participant,"new participant, development household",new participant and household,households in 2010 cohort,of which also in 2000 cohort
0,6,14659,6982,719,6958,10893,5573
1,8,12564,6015,633,5916,9517,4928
